# Contamination Detection via Context (CoDeC) for *Alle mot 1*

This notebook applies **Contamination Detection via Context (CoDeC)** to the
Norwegian television game show *Alle mot 1*. It follows the method introduced
by Zawalski et al. (2026) in [Detecting Data Contamination in LLMs via In-Context
Learning](https://arxiv.org/abs/2510.27055), adapting the authors'
[reference implementation](https://github.com/NVIDIA-NeMo/Evaluator) to the
open-weight `EleutherAI/pythia-1.4b-deduped` model.

Pythia is loaded locally through Hugging Face Transformers. The model revision
is pinned to improve reproducibility.

For a full explanation of the CoDeC method and its application to *Alle mot 1*,
see [`codec_qwen.ipynb`](codec_qwen.ipynb). The present notebook uses the same
data loading, CoDeC calculation, context sampling, aggregation, and reporting.


## Setup

In [ ]:
%pip install -q supabase transformers accelerate

import os
import random
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from supabase import create_client
from transformers import AutoModelForCausalLM, AutoTokenizer


## Configuration



In [ ]:
model_id = "EleutherAI/pythia-1.4b-deduped"
model_revision = "554d9c1bae3877f740aece41feb90f912cf9fedc"
questions_table = "codec_questions"

# Each target question is compared by adding one context example.
# Since the context is sampled randomly, the value of Δ(x) is subject to some variance.
# To improve stability, we average the Δ(x) values over 5 seeds.
num_context_examples = 1
n_seeds = 5
seed = 42
token_range = (10, -1)

supabase_url = os.environ.get("SUPABASE_URL")
supabase_key = os.environ.get("SUPABASE_KEY")

if not supabase_url or not supabase_key:
    raise ValueError("Missing Supabase environment variables.")
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-enabled GPU is required to run Pythia.")

db = create_client(supabase_url, supabase_key)


## Load Data
The query loads question transcripts and their identifying information from the database. Questions are ordered hierarchically by season, episode within season, and question number within episode.


In [ ]:
# The dataset contains fewer than Supabase's 1,000-row response limit.
columns = ["question_id", "season", "episode", "question_num", "transcript"]
fetch = (
    db.table(questions_table)
    .select(",".join(columns))
    .eq("include_for_codec", True)
    .eq("review_status", "READY")
    .order("season")
    .order("episode")
    .order("question_num")
    .limit(1000)
)
questions = pd.DataFrame(fetch.execute().data or [], columns=columns)

if questions[["question_id", "season", "transcript"]].isna().any().any():
    raise ValueError("Required question fields contain missing values.")

dataset_summary = (
    questions.assign(characters=questions["transcript"].str.len())
    .groupby("season", as_index=False)
    .agg(
        n_questions=("question_id", "size"),
        median_characters=("characters", "median"),
    )
)

print(f"Questions: {len(questions):,}")
display(dataset_summary)


## Core Implementation of CoDeC

The analysis consists of three components:

1. **Model handler:** Obtains tokens and their log probabilities from the language model.
2. **CoDeC detector:** Calculates the change in the target question's average log probability after adding context.
3. **Contamination detection pipeline:** Applies the CoDeC calculation to every question using five randomly sampled contexts and summarizes the results.

In [ ]:
class ModelHandler:
    """Loads a Hugging Face language model and returns token logprobs.

    The _extract method used in the ModelHandler for Qwen is removed. Pythia runs locally and returns logits directly,
    rather than returning a nested API response that must be unpacked.

    Methods:
    get_logprobs_and_tokens: Calculates logprobs for the observed tokens in text.
    """

    def __init__(
        self,
        model_name: str,
        revision: str,
        verbose: bool = False,
    ):
        self.model_name = model_name
        self.revision = revision
        self.device = torch.device("cuda")
        self.verbose = verbose

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            revision=revision,
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            revision=revision,
            torch_dtype=torch.float16,
        ).to(self.device)

        # Evaluation mode disables training-specific behavior.
        self.model.eval()

        # Store Pythia's context limit so overlength prompts stop with an error.
        self.max_length = int(self.model.config.max_position_embeddings)

        if self.verbose:
            print(
                f"Model ready: {model_name}\n"
                f"Revision: {revision}\n"
                f"Context window: {self.max_length:,} tokens"
            )

    def get_logprobs_and_tokens(
        self,
        text: str,
    ) -> tuple[np.ndarray, List[str]]:
        """
        Get logprobs and tokens for the observed tokens in input text.

        Args:
            text: Text for which token logprobs are calculated.

        Returns:
            Tuple containing the logprob array and matching token strings.
        """
        # Convert the question, or context-plus-question prompt, into token IDs.
        encoded = self.tokenizer(text, return_tensors="pt")
        input_ids = encoded["input_ids"]

        # shape[1] is the prompt length. No truncation is requested, so this
        # check raises an error instead of silently dropping overlength tokens.
        if input_ids.shape[1] > self.max_length:
            raise ValueError(
                f"Prompt has {input_ids.shape[1]:,} tokens, exceeding "
                f"the model's {self.max_length:,}-token context window."
            )

        input_ids = input_ids.to(self.device)

        # Inference mode avoids training and gradient calculations, reducing
        # memory use and speeding up the forward pass.
        with torch.inference_mode():
            logits = self.model(input_ids=input_ids).logits

        # Fireworks returned logprobs directly. Pythia returns raw logits, so
        # log_softmax converts them into normalized token log probabilities.
        # Each position predicts the observed token at the following position.
        logprobs = torch.log_softmax(
            logits[:, :-1].float(),
            dim=-1,
        )
        observed_ids = input_ids[:, 1:]

        # At every position, select the logprob assigned to the token that
        # actually appeared, rather than retaining the full vocabulary.
        observed_logprobs = logprobs.gather(
            dim=-1,
            index=observed_ids.unsqueeze(-1),
        ).squeeze(-1)

        token_ids = observed_ids[0].cpu().tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(token_ids)
        return observed_logprobs[0].cpu().numpy(), tokens


In [ ]:
model_handler = ModelHandler(
    model_name=model_id,
    revision=model_revision,
    verbose=True,
)


In [ ]:
class CoDeC:
    """Calculates CoDeC score by comparing target text with and without added context."""

    def __init__(self, token_range: tuple = token_range):
        """
        Initialize the contamination detector.

        Args:
            token_range: Range of target tokens to consider.
        """
        self.token_range = token_range

    def detect_contamination(
        self,
        target_text: str,
        context_examples: List[str],
        model_handler: ModelHandler,
    ) -> float | None:
        """
        Detect contamination for a single target text and context draw.

        Args:
            target_text: Text sample to test for contamination.
            context_examples: Other samples from the same dataset used as context.
            model_handler: Model handler used to obtain prompt logprobs.

        Returns:
            CoDeC delta (context minus baseline). A negative value is a
            contamination signal.
        """
        # Get target log probabilities without context.
        baseline_logprobs, baseline_tokens = (
            model_handler.get_logprobs_and_tokens(target_text)
        )
        target_length = len(baseline_logprobs)
        if target_length == 0:
            return None

        # Create context by joining examples.
        if not context_examples:
            raise ValueError("At least one context example is required.")
        context = "\n\n".join(context_examples)
        prompt_with_context = context + "\n\n" + target_text

        # Get target log probabilities after adding context.
        prompt_logprobs, prompt_tokens = (
            model_handler.get_logprobs_and_tokens(prompt_with_context)
        )
        if len(prompt_logprobs) < target_length:
            return None

        # The target appears at the end, so take the final target tokens.
        target_logprobs_after_context = prompt_logprobs[-target_length:]
        target_tokens_after_context = prompt_tokens[-target_length:]

        # Confirm that the same target tokens are compared in both conditions.
        if baseline_tokens != target_tokens_after_context:
            raise ValueError(
                "The target was not divided into the same tokens before "
                "and after adding context."
            )

        # Calculate average confidence for the specified token range.
        start_index, end_index = self.token_range
        if end_index == -1:
            end_index = target_length - 1
        else:
            end_index = min(end_index, target_length)

        if start_index >= end_index:
            print(f"Target text is too short: {target_text}")
            return None

        baseline = baseline_logprobs[start_index:end_index]
        after_context = target_logprobs_after_context[start_index:end_index]

        # Average the same finite target-token positions in both conditions.
        finite = np.isfinite(baseline) & np.isfinite(after_context)
        if not finite.any():
            return None

        confidence_baseline = np.mean(baseline[finite])
        confidence_after_context = np.mean(after_context[finite])

        # A negative difference means the target was easier without context.
        confidence_diff = confidence_after_context - confidence_baseline
        return float(confidence_diff)


In [ ]:
def contamination_detection_pipeline(
    model_handler: ModelHandler,
    dataset: pd.DataFrame,
    num_context_examples: int = 1,
    n_seeds: int = 5,
    seed: int = 42,
) -> Dict[str, Any]:
    """
    Run contamination detection on all season datasets.

    Args:
        model_handler: Model handler used for logprob inference.
        dataset: Dataframe with one row per question.
        num_context_examples: Number of same-season context questions.
        n_seeds: Number of random context draws per question.
        seed: Base random seed used to reproduce context selection.

    Returns:
        Dictionary with pooled, season, question, and context-draw results.
    """
    detector = CoDeC()
    rows = dataset.reset_index(drop=True).copy()


    rows["season_target_index"] = rows.groupby("season").cumcount()

    question_results = []
    seed_results = []
    print(
        f"Processing {len(rows)} questions across "
        f"{rows['season'].nunique()} seasons..."
    )

    # Process each question.
    for target_index in range(len(rows)):
        question_id = rows.loc[target_index, "question_id"]
        target_text = rows.loc[target_index, "transcript"]
        season = int(rows.loc[target_index, "season"])
        season_target_index = int(rows.loc[target_index, "season_target_index"])

        # Select context questions from the same season, excluding the target question.
        available_examples = rows.index[
            rows["season"].eq(season) & rows.index.to_series().ne(target_index)
        ].tolist()
        if not available_examples:
            raise ValueError(f"{question_id}: no same-season contexts are available.")

        deltas = []

        # Repeat with independently sampled contexts.
        for seed_index in range(n_seeds):
            random_seed = seed + season_target_index * 1000 + seed_index
            random_generator = random.Random(random_seed)

            if len(available_examples) >= num_context_examples:
                context_indices = random_generator.sample(
                    available_examples,
                    num_context_examples,
                )
            else:
                context_indices = available_examples

            context_examples = (
                rows.loc[context_indices, "transcript"].astype(str).tolist()
            )
            context_ids = (
                rows.loc[context_indices, "question_id"].astype(str).tolist()
            )

            delta = detector.detect_contamination(
                target_text,
                context_examples,
                model_handler,
            )
            if delta is not None and np.isfinite(delta):
                deltas.append(delta)
                seed_results.append({
                    "season": season,
                    "question_id": question_id,
                    "seed_index": seed_index,
                    "random_seed_used": random_seed,
                    "context_question_ids": "|".join(context_ids),
                    "delta": delta,
                })

        if len(deltas) != n_seeds:
            raise ValueError(
                f"{question_id}: expected {n_seeds} valid context draws, "
                f"but obtained {len(deltas)}."
            )

        # Average context draws before classifying the question.
        mean_delta = float(np.mean(deltas))
        question_results.append({
            "season": season,
            "question_id": question_id,
            "mean_delta": mean_delta,
            "delta_std": float(np.std(deltas, ddof=0)),
            "is_contaminated": mean_delta < 0,
            "n_seed_evals": len(deltas),
        })

    question_results = pd.DataFrame(question_results)
    seed_results = pd.DataFrame(seed_results)

    # Calculate one result per season.
    season_results = (
        question_results.groupby("season", as_index=False)
        .agg(
            n_questions=("question_id", "size"),
            codec_percent=("is_contaminated", lambda values: 100 * values.mean()),
            mean_delta=("mean_delta", "mean"),
            std_delta=("mean_delta", lambda values: values.std(ddof=0)),
        )
    )

    # Calculate one pooled result for the complete show.
    pooled_result = pd.DataFrame([{
        "season": "All seasons",
        "n_questions": len(question_results),
        "codec_percent": 100 * question_results["is_contaminated"].mean(),
        "mean_delta": question_results["mean_delta"].mean(),
        "std_delta": question_results["mean_delta"].std(ddof=0),
    }])
    summary_results = pd.concat(
        [pooled_result, season_results],
        ignore_index=True,
    )

    return {
        "summary_results": summary_results,
        "question_results": question_results,
        "seed_results": seed_results,
    }


## Run CoDeC

The pipeline performs five baseline/context comparisons per question, averages
the five deltas, and reports the pooled and season-level CoDeC scores.


In [ ]:
results = contamination_detection_pipeline(
    model_handler=model_handler,
    dataset=questions,
    num_context_examples=num_context_examples,
    n_seeds=n_seeds,
    seed=seed,
)

summary_results = results["summary_results"]
question_results = results["question_results"]
seed_results = results["seed_results"]

display(summary_results)


## Results

The left panel shows the distribution of five-draw mean deltas within each
season. The right panel pools the same question-level estimates across all
seasons. The red line marks the CoDeC decision threshold at zero.


In [ ]:
seasons = sorted(question_results["season"].unique())
season_deltas = [
    question_results.loc[
        question_results["season"].eq(season), "mean_delta"
    ].to_numpy()
    for season in seasons
]
pooled_deltas = question_results["mean_delta"].to_numpy()

fig, (ax_seasons, ax_pooled) = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
    gridspec_kw={"width_ratios": [2, 1]},
)

# Question-level delta distributions for each season.
positions = np.arange(1, len(seasons) + 1)
ax_seasons.boxplot(
    season_deltas,
    positions=positions,
    widths=0.55,
    showfliers=False,
)
jitter = np.random.default_rng(42)
for position, values in zip(positions, season_deltas):
    ax_seasons.scatter(
        jitter.normal(position, 0.05, len(values)),
        values,
        s=12,
        alpha=0.45,
    )
ax_seasons.axhline(0, color="#A23B3B", linewidth=1.2)
ax_seasons.set_xticks(positions, seasons)
ax_seasons.set_xlabel("Season")
ax_seasons.set_ylabel("Question-level mean delta")
ax_seasons.set_title("By season")

# Pooled distribution across all seasons.
ax_pooled.hist(pooled_deltas, bins=20, color="#6B8E7B", alpha=0.8)
ax_pooled.axvline(0, color="#A23B3B", linewidth=1.2)
ax_pooled.set_xlabel("Question-level mean delta")
ax_pooled.set_ylabel("Questions")
ax_pooled.set_title("All seasons")

pooled_score = 100 * question_results["is_contaminated"].mean()
ax_pooled.text(
    0.97,
    0.95,
    f"CoDeC: {pooled_score:.1f}%\nn = {len(question_results)}",
    transform=ax_pooled.transAxes,
    ha="right",
    va="top",
)

for axis in (ax_seasons, ax_pooled):
    axis.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"CoDeC results: {model_id.split('/')[-1]}")
fig.tight_layout()
plt.show()
